# Entomology Label OCR Pipeline

**Data:** CAS (California Academy of Sciences) arthropod specimens via iDigBio — same label format as ESSIG, publicly accessible.

**Flow:** iDigBio API → download specimen images → Roboflow finds text bounding boxes → Tesseract OCRs each crop → Darwin Core fields → CSV


## 1. Install

In [ ]:
%%capture
!pip install inference-sdk opencv-python-headless pytesseract requests pandas tqdm
!apt-get install -y tesseract-ocr tesseract-ocr-eng > /dev/null 2>&1
print('Done')

## 2. Config

In [ ]:
import os
ROBOFLOW_API_KEY     = 'jwdgcnRIwOz0kPB4Tpto'
ROBOFLOW_MODEL_ID    = 'entamology/6'
ROBOFLOW_API_URL     = 'https://serverless.roboflow.com'
DOWNLOAD_DIR         = '/content/cas_labels'
MAX_IMAGES           = 100
CONFIDENCE_THRESHOLD = 0.35
OCR_CONFIG           = r'--oem 3 --psm 6'
INFER_TIMEOUT        = 20
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
image_records = []
print('Config ready')

## 3. Fetch CAS Specimen Images via iDigBio

In [ ]:
import requests, time, re
from pathlib import Path
from tqdm.auto import tqdm

BASE = 'https://search.idigbio.org/v2'

def fetch_and_download(max_images=MAX_IMAGES, download_dir=DOWNLOAD_DIR):
    rq = {'hasImage': True, 'institutioncode': 'cas', 'phylum': 'arthropoda'}
    records = []
    offset, limit = 0, min(max_images, 100)
    print('Fetching CAS arthropod specimens with images...')

    while len(records) < max_images:
        payload = {'rq': rq, 'limit': limit, 'offset': offset}
        try:
            d = requests.post(f'{BASE}/search/records', json=payload, timeout=20).json()
        except Exception as e:
            print(f'API error: {e}'); break
        items = d.get('items', [])
        total = d.get('itemCount', 0)
        if not items: break

        for item in items:
            idx = item.get('indexTerms', {})
            media_uuids = idx.get('mediarecords', [])
            if not media_uuids: continue
            try:
                mr = requests.get(f'{BASE}/view/mediarecords/{media_uuids[0]}', timeout=10)
                url = mr.json().get('indexTerms', {}).get('accessuri', '')
            except:
                continue
            if not url: continue
            records.append({
                'uuid':           item.get('uuid', ''),
                'image_url':      url,
                'catalog_number': idx.get('catalognumber', item.get('uuid', '')),
                'taxon':          idx.get('scientificname', ''),
                'locality':       idx.get('locality', ''),
                'state':          idx.get('stateprovince', ''),
                'date':           idx.get('datecollected', ''),
                'collector':      idx.get('collector', ''),
            })
            if len(records) >= max_images: break

        print(f'  {len(records)}/{min(max_images, total)} (total available: {total})')
        if offset + limit >= total: break
        offset += limit
        time.sleep(0.3)

    print(f'Downloading {len(records)} images...')
    ready = []
    for rec in tqdm(records):
        name = re.sub(r'[^\w.-]', '_', str(rec['catalog_number']))[:80]
        dest = Path(download_dir) / f'{name}.jpg'
        if not dest.exists():
            try:
                resp = requests.get(rec['image_url'], timeout=20, allow_redirects=True)
                resp.raise_for_status()
                dest.write_bytes(resp.content)
            except Exception as e:
                print(f'  Skip {name}: {e}'); continue
            time.sleep(0.2)
        rec['local_path'] = str(dest)
        ready.append(rec)

    print(f'{len(ready)} images ready in {download_dir}')
    return ready


image_records = fetch_and_download()
print(f'Total: {len(image_records)}')

## 4. Roboflow Connection Test

In [ ]:
import cv2, signal
from inference_sdk import InferenceHTTPClient

CLIENT = InferenceHTTPClient(api_url=ROBOFLOW_API_URL, api_key=ROBOFLOW_API_KEY)
assert image_records, 'No images -- check cell 3'

test_path = image_records[0]['local_path']
print(f'Testing on: {test_path}')
try:
    result = CLIENT.infer(test_path, model_id=ROBOFLOW_MODEL_ID)
    preds  = result.get('predictions', [])
    print(f'SUCCESS: {len(preds)} text-region box(es) returned')
    for p in preds[:5]:
        print(f"  class={p.get('class')} conf={p.get('confidence'):.2f} "
              f"w={p.get('width')} h={p.get('height')}")
    if not preds:
        print('0 boxes -- lower CONFIDENCE_THRESHOLD in cell 2')
    print('\nFull result:', result)
except Exception as e:
    print(f'FAILED: {e}')

## 5. Detection + OCR

In [ ]:
import numpy as np, pytesseract
from google.colab.patches import cv2_imshow

class InferenceTimeout(Exception): pass
def _timeout_handler(signum, frame): raise InferenceTimeout()

def safe_infer(path):
    signal.signal(signal.SIGALRM, _timeout_handler)
    signal.alarm(INFER_TIMEOUT)
    try:
        r = CLIENT.infer(path, model_id=ROBOFLOW_MODEL_ID)
        signal.alarm(0); return r
    except InferenceTimeout:
        print('  [timeout]'); return None
    except Exception as e:
        signal.alarm(0); print(f'  [error] {e}'); return None

def preprocess_crop(crop):
    h, w = crop.shape[:2]
    scale = max(1, 1200 // max(h, w, 1))
    if scale > 1:
        crop = cv2.resize(crop, (w*scale, h*scale), interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    binary = cv2.adaptiveThreshold(gray, 255,
                 cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 15)
    return cv2.fastNlMeansDenoising(binary, h=10)

def detect_and_ocr(image_path, visualize=False):
    img = cv2.imread(image_path)
    if img is None: return []
    result      = safe_infer(image_path)
    predictions = result.get('predictions', []) if result else []
    boxes = []
    for i, pred in enumerate(predictions):
        conf = pred.get('confidence', 0)
        if conf < CONFIDENCE_THRESHOLD: continue
        cx, cy = int(pred['x']), int(pred['y'])
        bw, bh = int(pred['width']), int(pred['height'])
        cls    = pred.get('class', 'text')
        x1 = max(cx-bw//2, 0);  y1 = max(cy-bh//2, 0)
        x2 = min(cx+bw//2, img.shape[1]);  y2 = min(cy+bh//2, img.shape[0])
        crop = img[y1:y2, x1:x2]
        if crop.size == 0: continue
        raw_text = pytesseract.image_to_string(
            preprocess_crop(crop), config=OCR_CONFIG).strip()
        color = (0, 200, 0) if conf >= 0.7 else (0, 165, 255)
        cv2.rectangle(img, (x1,y1), (x2,y2), color, 2)
        cv2.putText(img, f'{cls} {conf:.2f}', (x1, max(y1-6,0)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
        boxes.append({'box_index':i, 'class':cls, 'confidence':round(conf,3),
                      'bbox':[x1,y1,x2,y2], 'raw_text':raw_text})
    if not boxes:
        raw = pytesseract.image_to_string(preprocess_crop(img), config=OCR_CONFIG).strip()
        boxes = [{'box_index':-1, 'class':'full_image', 'confidence':0.0,
                  'bbox':[], 'raw_text':raw}]
    if visualize:
        print(f'  {len(boxes)} text region(s)')
        cv2_imshow(img)
    return boxes

print('Detection + OCR ready')

## 6. Parse OCR Text → Darwin Core Fields

In [ ]:
import re
US_STATES = {'CA','CALIFORNIA','OR','OREGON','WA','WASHINGTON','AZ','ARIZONA',
             'NV','NEVADA','CO','COLORADO','TX','TEXAS','NM','NEW MEXICO',
             'UT','UTAH','ID','IDAHO','MT','MONTANA','WY','WYOMING'}

def _clean(t): return re.sub(r'\s+',' ',t).strip()

def parse_label_text(raw):
    j = ' '.join(l.strip() for l in raw.splitlines() if l.strip())
    o = dict(catalogNumber='', scientificName='', locality='', stateProvince='',
             country='', county='', eventDate='', recordedBy='',
             verbatimElevation='', samplingProtocol='', verbatimText=raw)
    m = re.search(r'\b(EMEC|CAS|CASENT|CIS)\s*([\d]+)\b', j, re.I)
    if m: o['catalogNumber'] = m.group(0).replace(' ','')
    m = re.search(r'\b([A-Z][a-z]{2,})\s+([a-z]{2,})\b', j)
    if m: o['scientificName'] = f'{m.group(1)} {m.group(2)}'
    for pat in [r'(\d{1,2})\s+([A-Za-z]{2,4})\s+(\d{4})',
                r'([A-Za-z]{3,4})\.?\s+(\d{1,2})[,\s]+(\d{4})',
                r'(\d{1,2})[/-](\d{1,2})[/-](\d{2,4})']:
        m = re.search(pat, j)
        if m: o['eventDate'] = _clean(m.group(0)); break
    m = re.search(r'(?:elev\.?\s*)?(\d{2,5})\s*(?:m|ft|feet|meters)', j, re.I)
    if m: o['verbatimElevation'] = _clean(m.group(0))
    m = re.search(r'(?:leg|coll?|collector)\.?\s*([A-Z][a-zA-Z.\s&]{3,40})', j)
    if m: o['recordedBy'] = _clean(m.group(1))
    for kw in ['Malaise','flight intercept','pitfall','light trap','sweep','Berlese','hand collect']:
        if re.search(kw, j, re.I): o['samplingProtocol'] = kw; break
    for state in US_STATES:
        if re.search(r'\b'+state+r'\b', j, re.I):
            o['stateProvince'] = state.title(); o['country'] = 'USA'; break
    m = re.search(r'([A-Z][a-zA-Z]+(?:\s[A-Z][a-zA-Z]+)?)\s+(?:Co\.|County)', j)
    if m: o['county'] = m.group(1)
    loc = re.sub(r'(?:EMEC|CAS|CASENT|CIS)[\s\d]+|leg\.?|coll?\.?|\d{4}|\bCo\.|County',
                 '', j, flags=re.I)
    o['locality'] = _clean(loc)[:120]
    return o

print('Parser ready')

## 7. Run Full Pipeline

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

assert image_records, 'No images -- check cell 3'

VISUALIZE_FIRST_N = 3
all_rows = []

for idx, rec in enumerate(tqdm(image_records, desc='Images')):
    path = rec['local_path']
    show = idx < VISUALIZE_FIRST_N
    if show: print(f'\n-- {rec["catalog_number"]} | {rec["taxon"]} --')
    boxes = detect_and_ocr(path, visualize=show)
    for box in boxes:
        parsed = parse_label_text(box['raw_text'])
        if not parsed['catalogNumber']: parsed['catalogNumber'] = rec['catalog_number']
        all_rows.append({
            'source_image':   path,
            'idigbio_uuid':   rec.get('uuid',''),
            'gbif_taxon':     rec.get('taxon',''),
            'gbif_locality':  rec.get('locality',''),
            'gbif_state':     rec.get('state',''),
            'gbif_date':      rec.get('date',''),
            'gbif_collector': rec.get('collector',''),
            'box_index':      box['box_index'],
            'box_class':      box['class'],
            'box_conf':       box['confidence'],
            **parsed
        })
        if show:
            print(f"  [{box['class']} conf={box['confidence']}] {box['raw_text'][:100]}")
            print(f"  -> date={parsed['eventDate']} | state={parsed['stateProvince']} | collector={parsed['recordedBy']}")

df = pd.DataFrame(all_rows)
print(f'\nDone: {len(image_records)} images, {len(df)} label records')
display(df.head(10))

## 8. Quality Report

In [ ]:
dc_fields = ['scientificName','locality','stateProvince','eventDate',
             'recordedBy','verbatimElevation','samplingProtocol']
for f in dc_fields:
    if f not in df.columns: df[f] = ''
print('Darwin Core OCR fill rates:')
for f in dc_fields:
    filled = df[f].astype(bool).sum()
    pct    = 100 * filled / max(len(df), 1)
    bar    = chr(9608)*int(pct//5) + chr(9617)*(20-int(pct//5))
    print(f'  {f:<22} {bar} {pct:5.1f}%')
print('\nTop states:')
print(df['stateProvince'].value_counts().head(8).to_string())
print('\nTop taxa:')
print(df['scientificName'].value_counts().head(8).to_string())

## 9. Export CSV

In [ ]:
from google.colab import files
out = '/content/cas_labels_parsed.csv'
df.to_csv(out, index=False)
print(f'Saved {len(df)} rows')
files.download(out)